## This notebook can be used to rank a list of nodes from a category that connect to an entity such as a gene. 

## Quick start: `neighborhood_finder` API

TCT now provides a developer-friendly `neighborhood_finder` wrapper as part of the main API. It accepts labels or CURIEs (single or list), resolves and normalizes inputs, caches Translator resources, and returns a `FinderResult`. Short category names like `"Disease"` are converted to `"biolink:Disease"` automatically. Advanced controls: `node_categories`, `predicates_subset`, `attribute_constraints`, `resources`.


In [1]:
from TCT import get_translator_resources, neighborhood_finder

resources = get_translator_resources()
neighbors = neighborhood_finder(
    node="MONDO:0004979",
    neighbor_categories=["SmallMolecule", "Drug"],
    node_categories=["Disease"],
    resources=resources,
)
{
    "node_count": len(neighbors.knowledge_graph.get("nodes", {})),
    "edge_count": len(neighbors.knowledge_graph.get("edges", {})),
    "result_count": len(neighbors.results),
}


Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}
59
(18613, 5)
(30172, 5)
Automat-monarchinitiative(Trapi v1.5.0): Success!
COHD TRAPI: Success!
Automat-icees-kg(Trapi v1.5.0): Success!
RTX KG2 - TRAPI 1.5.0: Success!
Retriever: Success!
Automat-robokop(Trapi v1.5.0): Success!
BioThings Explorer (BTE) TRAPI: Success!
MolePro: Success!
ARAX Translator Reasoner - TRAPI 1.6.0: Success!


{'node_count': 3984, 'edge_count': 12283, 'result_count': 1}

In [5]:
# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_' + timestamp + '.json', 'w') as f:
    json.dump(neighbors.raw, f)

# The result can be visualized in visualize_TCT_results.html

In [10]:
constrained = neighborhood_finder(
    node="MONDO:0004979",
    neighbor_categories=["SmallMolecule","Drug"],
    node_categories=["Disease"],
    predicates_subset=["biolink:treats", "biolink:ameliorates"],
    attribute_constraints=[
        {
            "id": "biolink:knowledge_level",
            "operator": "==",
            "value": "knowledge_assertion",
        }
    ],
    resources=resources,
)


In [11]:
# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_' + timestamp + '.json', 'w') as f:
    json.dump(constrained.raw, f)

# The result can be visualized in visualize_TCT_results.html

## Customized Usage

Sometimes we may want to run neighborhood finder with certain API endpoints.

In [12]:

import sys
import os
sys.path.append('../TCT/')
from TCT import node_normalizer
from TCT import name_resolver
from TCT import translator_metakg
from TCT import translator_kpinfo
from TCT import translator_query
from TCT import TCT_neighborhood_finder

from TCT import TCT



### Load Translator resources


In [13]:
APInames, metaKG, Translator_KP_info= translator_metakg.load_translator_resources(use_new_metakg_url=True)

All_predicates = list(set(metaKG['Predicate']))
All_categories = list((set(list(set(metaKG['Subject']))+list(set(metaKG['Object'])))))
API_withMetaKG = list(set(metaKG['API']))

API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(metaKG[metaKG['API'] == api]['Predicate']))

Skipping server without x-maturity: {'url': '/sipr'}
Skipping server without x-maturity: {'description': 'Local dev', 'url': 'http://127.0.0.1:5001'}


### Select endpoints for query


In [14]:
# This is an example of selecting a list of APIs for the neighborhood finder. The user can modify this list to include the APIs they want to use. The APIs in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. The user can also modify the list of predicates to use for finding the neighborhood. The predicates in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph. 
# The user can also modify the list of categories to use for finding the neighborhood. 
# The categories in this list are the ones that will be used to find the neighborhood of a given node in the knowledge graph.
# if selected_APIlist is empty, use all APIs in APInames
selected_APIlist = ['Retriever',
                    #'Clinical Trials KP - TRAPI 1.5.0',
                    #'Drug Approvals KP - TRAPI 1.5.0',
                    'Genetics Data Provider for NCATS Biomedical Translator Reasoners',
                    #'Microbiome KP - TRAPI 1.5.0',
                    #'MolePro',
                    #'COHD TRAPI',
                    #'RTX KG2 - TRAPI 1.5.0',
                    #'Text Mined Cooccurrence API',
                    'CATRAX BigGIM DrugResponse Performance Phase KP - TRAPI 1.5.0',
                    'CATRAX Pharmacogenomics KP - TRAPI 1.5.0',
                    ]

# add Automat API to the selected API list if it is not already in the list
#for api in APInames:
#    if 'Automat' in api and api not in selected_APIlist:
#        selected_APIlist.append(api)
        
#selected_APIlist = ['Retriever'] # select just Retriever endpoint
# select a list of APIs to use and a list of predicates to use
if len(selected_APIlist) == 0:
    select_APIs = APInames
else:
    select_APIs = {k: APInames[k] for k in selected_APIlist if k in APInames}


selected_metaKG = metaKG[metaKG['API'].isin(select_APIs.keys())]
#print(select_APIs)


All_predicates = list(set(selected_metaKG['Predicate']))
All_categories = list((set(list(set(selected_metaKG['Subject']))+list(set(selected_metaKG['Object'])))))
API_withMetaKG = list(set(selected_metaKG['API']))
API_predicates = {}
for api in API_withMetaKG:
    API_predicates[api] = list(set(selected_metaKG[selected_metaKG['API'] == api]['Predicate']))

## Find the neighborhood of an entity from a subset of APIs 


In [15]:
#name_resolver.lookup('BACE1', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
#name_resolver.lookup('NPM1', return_top_response=False, biolink_type='biolink:Gene',  limit=100, only_taxa='NCBITaxon:9606') # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('4q12 microdeletion syndrome')
#name_resolver.lookup('CDK9', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')
#name_resolver.lookup('alzheimer disease', return_top_response=False, biolink_type='biolink:Disease',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('Penicillamine')
#name_resolver.lookup('Penicillamine', return_top_response=True, biolink_type='biolink:Drug',  limit=100) # sometimes the identifiers are not in the top 1, users need to check the other returned results

#name_resolver.lookup('acute myeloid leukemia', return_top_response=True, biolink_type='biolink:Disease',  limit=10) # sometimes the identifiers are not in the top 1, users need to check the other returned results
#name_resolver.lookup('CDK9', return_top_response = False)

#name_resolver.lookup('MYB', only_taxa='NCBITaxon:9606', biolink_type='biolink:Gene')

name_resolver.lookup('ovarian cancer', return_top_response=True, biolink_type='biolink:Disease',  limit=10) 

TranslatorNode(curie='MONDO:0008170', label='ovarian cancer', types=['biolink:Disease', 'biolink:DiseaseOrPhenotypicFeature', 'biolink:BiologicalEntity', 'biolink:ThingWithTaxon', 'biolink:NamedThing', 'biolink:Entity'], synonyms=None, curie_synonyms=None, attributes=None, taxa=[])

In order to use the neighborhood finder, we have to look up a CURIE ID for a given term.

In [17]:

#input_identifiers = 'MONDO:0004975'
#
input_identifiers = 'MONDO:0016833'
#input_identifiers = 'CHEBI:145499'
input_identifiers = 'MONDO:0004975'
#input_identifiers = 'NCBIGene:1956'
#input_node_info = node_normalizer.get_normalized_nodes(input_identifiers)
#input_node_info
input_identifiers = "MONDO:0016833"
input_identifiers = "NCBIGene:4869"
#input_identifiers = 'MONDO:0018874'
input_identifiers = "NCBIGene:4869"
input_identifiers = 'MONDO:0016833' # 14q12 microdeletion syndrome 
input_identifiers = 'NCBIGene:2290' # FOXG1
input_identifiers = "NCBIGene:6261"
input_identifiers = "MONDO:0004975" # Alzheimer's disease
input_identifiers = "MONDO:0008170" # 14q12 microde


Neighborhood finder identifies all nodes *b* that are connected to the given node *a*, where *b* is part of a defined list of categories - returning the neighborhood of node *a*.

In [18]:
# to exclude BioThings Explorer (BTE) TRAPI:
finder_result = TCT_neighborhood_finder.neighborhood_finder(
    input_identifiers,
    # neighbor_categories=['biolink:Drug', 'biolink:SmallMolecule', 'biolink:ChemicalSubstance'],
    # neighbor_categories=['biolink:AnatomicalEntity'],
    neighbor_categories=['biolink:Gene'],
    api_names=select_APIs,
    meta_kg=selected_metaKG,
    api_predicates=API_predicates,
    node_categories=['biolink:Gene'],
)
TCT_neighborhood_finder_result = finder_result.raw

# write a result to a json file
import json
# add a timestamp to the file name
import datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
with open('TCT_neighborhood_finder_result_' + input_identifiers.replace(':', '_') + '_' + timestamp + '.json', 'w') as f:
    json.dump(TCT_neighborhood_finder_result, f)


CATRAX Pharmacogenomics KP - TRAPI 1.5.0: Success!
Retriever: Success!
